In [1]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf
import sys
import importlib


In [2]:


fixed_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
#utils_path = 'C:\\Users\\Enrico Didoli\\0.Master_Thesis\\'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

general_functions_path = f'{fixed_path}General_Functions/'
if general_functions_path not in sys.path:
    sys.path.append(general_functions_path)

# Aggiungi la directory principale del progetto al percorso di sistema
csnn_path = f'{fixed_path}CSNN\\csnn\\'
if csnn_path not in sys.path:
    sys.path.append(csnn_path)


path_model_unfixed = f'{csnn_path}0.Unfixed_files'
if path_model_unfixed  not in sys.path:
    sys.path.append(path_model_unfixed )
# Ricaricali per forzare la lettura aggiornata del file


In [3]:
decache_files = ['timepoints_elaboration', 'results_elaboration', 'new_datasets_generation']


# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)


from utils import trainer, DensityDifference, dataset

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv


from timepoints_elaboration import donation_extraction, dataset_elaboration, donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from new_datasets_generation import splitting_and_dataset_elaboration, remove_labels


timepoints_elaboration rimosso dalla cache
results_elaboration non trovato nella cache
new_datasets_generation non trovato nella cache


In [4]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    data = pd.read_csv(file_path, sep = ';', decimal = ',').astype('float32')
    ALL_DATASETS.append(data) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for data in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [data]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [5]:
for i, data in enumerate(ALL_DATASETS):
    blast_n = (data['IsBlast'] == 1).sum()
    print(f'data {i} has {blast_n} cells over {len(data)} cells')

data 0 has 0 cells over 6558750 cells
data 1 has 0 cells over 2764877 cells
data 2 has 0 cells over 1291729 cells
data 3 has 1348 cells over 843500 cells
data 4 has 170 cells over 1404000 cells
data 5 has 0 cells over 3265250 cells
data 6 has 639 cells over 430869 cells
data 7 has 15 cells over 722000 cells
data 8 has 757 cells over 864000 cells
data 9 has 0 cells over 1947518 cells
data 10 has 308059 cells over 778750 cells
data 11 has 830101 cells over 5510750 cells
data 12 has 72 cells over 208000 cells
data 13 has 0 cells over 2912500 cells
data 14 has 3096 cells over 160500 cells
data 15 has 1449 cells over 3591480 cells
data 16 has 0 cells over 3107000 cells
data 17 has 9147 cells over 637409 cells
data 18 has 227 cells over 2928000 cells
data 19 has 518 cells over 164553 cells
data 20 has 398 cells over 191975 cells
data 21 has 0 cells over 169000 cells
data 22 has 2390 cells over 479000 cells
data 23 has 77 cells over 455000 cells
data 24 has 44697 cells over 160828 cells
data 

In [6]:
# Show patients donations
print(multiple_donations)
#['12', '9', '13', '7', '11'] 

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [7]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [8]:

# Samples donors for Train, Validation and Test sets    
train_donors_idx, val_donors_idx, test_donors_idx = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, seed = 10)



Precess starts. Dividing donors...
healthy_donors_idx, blast_donors_idx, mixed_donors_idx: [4, 0, 1, 2, 3], [0, 1, 2],[2, 1, 0, 3, 4]
Seting Train, Validation and Test idx...
['9', '12', '13', '3'] ['2', '14', '11'] ['6', '8', '15', '1', '4', '7']


In [9]:

#  Retrieves specific donor datasets from all datasets list
train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
test_datasets_extracted = donation_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

#print(len(train_datasets_extracted)) # list of donators
#print(len(train_datasets_extracted[0])) # list of donations
#print(len(train_datasets_extracted[0][0])) # dataset of cells

In [10]:
n_sub = 3
seed = 42
n_cells = 100000
new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y = splitting_and_dataset_elaboration(train_datasets_extracted, val_datasets_extracted, test_datasets_extracted, n_sub, n_cells, seed, cv = True)

cv_train_datasets = new_train_datasets + new_val_datasets
cv_train_y = new_train_y + new_val_y

new_no_label_test_dataset = remove_labels(new_test_datasets)
print(len(cv_train_datasets))

New training datasets creation...
4
2

New Donor
2
Tot blast data in the donor timepoints: 44697
Tot blast data in the donor timepoints: 46110
Extraction Done
Condition: 1
Chosen # of blast cells: [20000.   500. 10000.]
Chosen # of blast cells: [  500. 10000. 20000.]

New Donor
2
Tot blast data in the donor timepoints: 639
Tot blast data in the donor timepoints: 654
Extraction Done
Condition: 1
Chosen # of blast cells: [10000. 10000.  1000.]
Chosen # of blast cells: [ 1000. 10000. 10000.]

New Donor
1
Timepoint condition: Healthy
Extraction Done
Condition: 0

New Donor
2
Tot blast data in the donor timepoints: 72
Tot blast data in the donor timepoints: 72
Extraction Done
Condition: 1
Chosen # of blast cells: [10000.  5000. 20000.]
Chosen # of blast cells: [ 5000. 10000. 20000.]
[1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0]
Done!

New training datasets creation...

New Donor
Tot blast data in the donor timepoints: 308059
Tot blast data in the donor timepoints: 1138160


In [11]:
print(len(cv_train_datasets))
print(len(cv_train_y))
print(cv_train_y)


7
7
[[1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [0, 0, 0], [1, 1, 1, 0, 0, 0]]


In [12]:
for i, dataset in enumerate(new_test_datasets):
    blast_n = (dataset['IsBlast'] == 1).sum()
    
    print(f'dataset {i} has {blast_n} cells over {len(dataset)} cells. Label: {new_test_y[i]}')

dataset 0 has 500 cells over 100000 cells. Label: 1
dataset 1 has 1000 cells over 100000 cells. Label: 1
dataset 2 has 1000 cells over 100000 cells. Label: 1
dataset 3 has 0 cells over 100000 cells. Label: 0
dataset 4 has 0 cells over 100000 cells. Label: 0
dataset 5 has 0 cells over 100000 cells. Label: 0
dataset 6 has 1000 cells over 100000 cells. Label: 1
dataset 7 has 10000 cells over 100000 cells. Label: 1
dataset 8 has 20000 cells over 100000 cells. Label: 1
dataset 9 has 0 cells over 100000 cells. Label: 0
dataset 10 has 0 cells over 100000 cells. Label: 0
dataset 11 has 0 cells over 100000 cells. Label: 0
dataset 12 has 0 cells over 100000 cells. Label: 0
dataset 13 has 0 cells over 100000 cells. Label: 0
dataset 14 has 0 cells over 100000 cells. Label: 0
dataset 15 has 500 cells over 100000 cells. Label: 1
dataset 16 has 500 cells over 100000 cells. Label: 1
dataset 17 has 10000 cells over 100000 cells. Label: 1
dataset 18 has 0 cells over 100000 cells. Label: 0
dataset 19 has

In [13]:
print(len(cv_train_datasets))
print(len(new_val_datasets))
print(len(new_test_datasets))

7
3
33


In [14]:
import time
start = time.time()
for i, data in enumerate(cv_train_datasets):
    start = time.time()
    for sub in data:
        for cell in sub:
            for j, data1 in enumerate(new_test_datasets):
                if j != i:
                    #for sub_d in data1:
                        if cell in data1.values:
                            print('ok')
    end = time.time() - start
    print(end)

0.040761709213256836
0.02199864387512207
0.00634455680847168
0.015630722045898438
0.02870631217956543
0.004021883010864258
0.03240847587585449


C:\Users\admin\AppData\Local\Temp\ipykernel_12924\1854527084.py:10: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if cell in data1.values:


In [15]:

from utils import trainer, DensityDifference, dataset


decache_files = ['tuning_ball_logistic.yaml', 'timepoints_elaboration', 'results_elaboration', 'new_datasets_generation',
                'utils.trainer', 'utils.DensityDifference', 'utils.dataset']


# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)

from utils import trainer, DensityDifference, dataset

from timepoints_elaboration import donation_extraction, dataset_elaboration, donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from new_datasets_generation import splitting_and_dataset_elaboration, remove_labels


tuning_ball_logistic.yaml non trovato nella cache
timepoints_elaboration rimosso dalla cache
results_elaboration rimosso dalla cache
new_datasets_generation rimosso dalla cache
utils.trainer rimosso dalla cache
utils.DensityDifference rimosso dalla cache
utils.dataset rimosso dalla cache


In [16]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import os


RESULT_DIR = f'{path_model_unfixed}/data/B-ALL'

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(f'{RESULT_DIR}/train', exist_ok=True)
os.makedirs(f'{RESULT_DIR}/test', exist_ok=True)

#train_set = pd.read_csv("metadata/B-ALL_train.csv")
#test_set = pd.read_csv("metadata/B-ALL_test.csv")
#print(len(new_train_datasets))
#print(len(new_train_y))

idx = 0
for j, patient in enumerate(cv_train_datasets):
    for i, data in enumerate(patient):
        #change indexing to allow different patients
        idx += 1

        feature_names = data.columns
    
        b_data = len(data[data['IsBlast'] == 1])
        proportion = b_data / len(data)
        data = data.drop(columns = ['IsBlast'])
        
        X = torch.tensor(data.to_numpy())
        
        y = cv_train_y[j][i]
        #print(idx)
        data = {
            "X": X,
            "y": y,
            "idx": idx,
            "proportion": proportion,
            "tags": ["B-ALL", "train"],
            "feature_names": feature_names
        }
        path = f'{RESULT_DIR}/train/'+ f"{j}_{idx}.pt"
    
        torch.save(data, path)

Index = 0
for i, data in enumerate(new_test_datasets):
    Index += 1
    feature_names = data.columns

    b_data = len(data[data['IsBlast'] == 1])
    proportion = b_data / len(data)
    data = data.drop(columns = ['IsBlast'])
    
    X = torch.tensor(data.to_numpy())
    #X = torch.tensor(data)
    print(X.shape)
    y = new_test_y[i]
    
    data = {
        "X": X,
        "y": y,
        "idx": Index,
        "proportion": proportion,
        "tags": ["B-ALL", "test"],
        "feature_names": feature_names
    }
    path = f'{RESULT_DIR}/test/'+ f"{Index}.pt"
    
    torch.save(data, path)

torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])


In [17]:
import sys
from yaml import load, FullLoader
sys.argv = ['tuning_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\tuning_ball_logistic.yaml']

In [18]:
%%time
import numpy as np
import random
import torch
import csv
import os
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import ParameterGrid
from yaml import load, FullLoader
import sys
from tqdm import tqdm

from utils.dataset import PointsetDataset
from utils.trainer import dd_initialize, train_initialize, train_head, finetune, load_evaluate

with open(sys.argv[1], 'r') as f:
    config = load(f, Loader=FullLoader) # load parameters set in configuficuration file .yaml 
    #print(config)

import os
print(f'{RESULT_DIR}/train')
print(os.path.exists(f'{RESULT_DIR}/train'))

start_seed = config['start_seed'] # retrieve the seed
end_seed = config['start_seed'] + config['n_seeds'] # retrieve the final seed

metrics = [
    ("accuracy", accuracy_score),
    ("f1", f1_score),
    ("precision", precision_score),
    ("recall", recall_score),
    ("auc", roc_auc_score),
]

param_grid = ParameterGrid(config['param_grid']) # makes all combinations of parameters (originally 80) for grid search
sample = param_grid[0] # start with the first combination

keys = list(sample.keys()) # retrieve oarameters names to use them as column name in the final results.csv file


# create the empty results.csv file where insert all parmaeters and metrics of the trial
os.makedirs(f"experiment/{config['experiment_name']}/", exist_ok=True)
with open(f"experiment/{config['experiment_name']}/results.csv", 'w') as f: #open the file as writer
    writer = csv.writer(f)
    row_names = ["seed"] 
    writer.writerow(row_names + list(sample.keys()) + ["param_hash"] +
                    [name + "_train" for name, _ in metrics] + 
                    [name + "_valid" for name, _ in metrics])


#set seed and device
np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


print(f'start_training...')
for seed in range(1): #range(start_seed, end_seed):

    # retrieve train and validation data (dataset.py)   
    dataset_train = PointsetDataset(config['datasets'], train=True, fold=5, random_state=seed, **config['dataset_config'])
    dataset_valid = PointsetDataset(config['datasets'], train=False, fold=5, random_state=seed, **config['dataset_config'])


    
    #**config['dataset_config'] -> ** operatore takes keys of a dictionary as parameters and set values as the value of those parameters
    # This means that config['dataset_config'] (which is the value of the key dataset_config in config dictionary) is a dictionary itself


    # obtain the initial scores using kernel density (training a KDE)
    diff, allpos_sample, allneg_sample = dd_initialize(dataset_train, dataset_valid, **config['experiment_config'])

    
    for params in tqdm(param_grid): # for every combination of parameters 

        # creates the unique code to save parameter and results in the repository
        hashparams = params.copy()
        hashparams['architecture'] = frozenset(hashparams['architecture'])
        param_hash = hash(frozenset(hashparams.items()))
        
        #results directory
        resdir = f"experiment/{ str(config['experiment_name']) }/models/seed_{ int(seed) }/hash_{ int(param_hash) }"
        os.makedirs(resdir, exist_ok=True)

        if True:
            kwargs = params.copy()
            print(f'kwargs: {kwargs}')

            # extract parameters from configuration files and use them in training 
            for key in config['experiment_config']:
                kwargs[key] = config['experiment_config'][key] #takes parameters from config and Adds them in kwargs dictionary
            print(f'kwargs_AFTER: {kwargs}')
            
            # train the model
            model, Xinit_train, yinit_train = train_initialize(dataset_train, dataset_valid, diff, 
                                                               allpos_sample, allneg_sample, **kwargs)

            print('here')
            torch.save((Xinit_train, yinit_train), f"{resdir}/dd_init.pt")
            torch.save(model.cpu(), f"{resdir}/model_init.pt")

            # train again the model with the best  parameters and weights? 
            model = train_head(dataset_train, dataset_valid, model, **config['experiment_config'])

            torch.save(model.cpu(), f"{resdir}/model_head.pt")

            
            model, (preds_val, y_valid), (preds_train, y_train) = finetune(dataset_train, dataset_valid, model, **kwargs)

            torch.save(model, f"{resdir}/model.pt")
        else:
            # validation
            model, (preds_val, y_valid), (preds_train, y_train) = load_evaluate(dataset_train, dataset_valid, "%s/model.pt")
            
        train_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                train_metrics.append(metric(y_train.numpy(), (preds_train > 0.5).to(int).numpy()))
            else:
                train_metrics.append(metric(y_train.numpy(), preds_train.numpy()))

        val_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                val_metrics.append(metric(y_valid.numpy(), (preds_val > 0.5).to(int).numpy()))
            else:
                val_metrics.append(metric(y_valid.numpy(), preds_val.numpy()))

        #params['hash'] = param_hash
        with open(f"experiment/{config['experiment_name']}/results.csv", 'a') as f:
            writer = csv.writer(f)
            metadata = [seed]
            param_list = [str(v) for v in params.values()]
            param_list.append(param_hash)
            writer.writerow(metadata + param_list + train_metrics + val_metrics)


C:\Users\admin\0.Master_Thesis\CSNN\csnn\0.Unfixed_files/data/B-ALL/train
True
start_training...
Train dataset has 6 files. Valid dataset has 36 files.
train self_indices: ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\3_17.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\1_12.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\0_1.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\1_9.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\4_26.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\0_6.pt']
Train dataset has 6 files. Valid dataset has 36 files.
valid self_indices: ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\3_19.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\3_21.pt', 'C:\\Users\\admin\

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 194.43it/s]


Iteration 0: shape=torch.Size([100000, 11])

perm1 100000
next: 10000
perm2 10000
perm_train: [26719 56479 99173 ... 87649 50722  8819]
perm_sample: [74192 81575 21868 ... 21657 28032 40173]
check ap_train : 5000
check ap_sample : 5000
Iteration 1: shape=torch.Size([100000, 11])

perm1 100000
next: 10000
perm2 10000
perm_train: [21212 90847 98957 ... 85617 73277 95142]
perm_sample: [25501 80213 17535 ...  5053 38811 94198]
check ap_train : 5000
check ap_sample : 5000
Iteration 2: shape=torch.Size([100000, 11])

perm1 100000
next: 10000
perm2 10000
perm_train: [59891 66817 29245 ... 82743 83345 92626]
perm_sample: [83774 97964 50976 ... 49486 67053 94590]
check ap_train : 5000
check ap_sample : 5000
all_neg2: 15000
15000
neg: 15000


  0%|                                                                                            | 0/8 [00:00<?, ?it/s]

kwargs: {'architecture': [3, 1], 'dd_threshold': 95, 'lr': 0.001, 'negative_penalty': 1}
kwargs_AFTER: {'architecture': [3, 1], 'dd_threshold': 95, 'lr': 0.001, 'negative_penalty': 1, 'alpha': 0.9, 'dd_sample_size': 5000, 'max_iter': 100000, 'bayesian_dd': True}
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.7510, tr_acc: 0.5000
[Epoch 250] tr_loss: 0.4072, tr_acc: 0.9413
[Epoch 500] tr_loss: 0.1277, tr_acc: 0.9927
[Epoch 750] tr_loss: 0.0526, tr_acc: 0.9973
[Epoch 1000] tr_loss: 0.0310, tr_acc: 0.9973
[Epoch 1250] tr_loss: 0.0219, tr_acc: 0.9973
[Epoch 1500] tr_loss: 0.0172, tr_acc: 0.9973
[Epoch 1750] tr_loss: 0.0144, tr_acc: 0.9973
[Epoch 2000] tr_loss: 0.0125, tr_acc: 0.9973
[Epoch 2250] tr_loss: 0.0112, tr_acc: 0.9973
[Epoch 2500] tr_loss: 0.0101, tr_acc: 0.9973
[Epoch 2750] tr_loss: 0.0092, tr_acc: 0.9980
[Epoch 3000] tr_loss: 0.0085, tr_acc: 0.9980
[Epoch 3250] tr_loss: 0.0079, tr_acc: 0.9980
[Epoch 3500] tr_loss: 0.0074, tr_acc: 0.9980
[Epoch 3750] tr_loss: 0.0069, tr

...

kwargs: {'architecture': [6, 1], 'dd_threshold': 99.9, 'lr': 0.001, 'negative_penalty': 1}
kwargs_AFTER: {'architecture': [6, 1], 'dd_threshold': 99.9, 'lr': 0.001, 'negative_penalty': 1, 'alpha': 0.9, 'dd_sample_size': 5000, 'max_iter': 100000, 'bayesian_dd': True}
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.7348, tr_acc: 0.2333
[Epoch 250] tr_loss: 0.2795, tr_acc: 0.9333
[Epoch 500] tr_loss: 0.0933, tr_acc: 1.0000
[Epoch 750] tr_loss: 0.0427, tr_acc: 1.0000
[Epoch 1000] tr_loss: 0.0230, tr_acc: 1.0000
[Epoch 1250] tr_loss: 0.0139, tr_acc: 1.0000
[Epoch 1500] tr_loss: 0.0090, tr_acc: 1.0000
[Epoch 1750] tr_loss: 0.0062, tr_acc: 1.0000
[Epoch 2000] tr_loss: 0.0045, tr_acc: 1.0000
[Epoch 2250] tr_loss: 0.0033, tr_acc: 1.0000
[Epoch 2500] tr_loss: 0.0025, tr_acc: 1.0000
[Epoch 2750] tr_loss: 0.0020, tr_acc: 1.0000
[Epoch 3000] tr_loss: 0.0016, tr_acc: 1.0000
[Epoch 3250] tr_loss: 0.0013, tr_acc: 1.0000
[Epoch 3500] tr_loss: 0.0010, tr_acc: 1.0000
[Epoch 3750] tr_loss: 0.0008

 88%|████████████████████████████████████████████████████████████████████████▋          | 7/8 [15:35<02:02, 122.06s/it]

kwargs: {'architecture': [6, 1], 'dd_threshold': 99.9, 'lr': 0.0001, 'negative_penalty': 1}
kwargs_AFTER: {'architecture': [6, 1], 'dd_threshold': 99.9, 'lr': 0.0001, 'negative_penalty': 1, 'alpha': 0.9, 'dd_sample_size': 5000, 'max_iter': 100000, 'bayesian_dd': True}
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.6566, tr_acc: 0.4333
[Epoch 250] tr_loss: 0.1540, tr_acc: 1.0000
[Epoch 500] tr_loss: 0.0460, tr_acc: 1.0000
[Epoch 750] tr_loss: 0.0204, tr_acc: 1.0000
[Epoch 1000] tr_loss: 0.0110, tr_acc: 1.0000
[Epoch 1250] tr_loss: 0.0067, tr_acc: 1.0000
[Epoch 1500] tr_loss: 0.0045, tr_acc: 1.0000
[Epoch 1750] tr_loss: 0.0031, tr_acc: 1.0000
[Epoch 2000] tr_loss: 0.0023, tr_acc: 1.0000
[Epoch 2250] tr_loss: 0.0017, tr_acc: 1.0000
[Epoch 2500] tr_loss: 0.0013, tr_acc: 1.0000
[Epoch 2750] tr_loss: 0.0010, tr_acc: 1.0000
[Epoch 3000] tr_loss: 0.0008, tr_acc: 1.0000
[Epoch 3250] tr_loss: 0.0007, tr_acc: 1.0000
[Epoch 3500] tr_loss: 0.0005, tr_acc: 1.0000
[Epoch 3750] tr_loss: 0.00

100%|███████████████████████████████████████████████████████████████████████████████████| 8/8 [16:03<00:00, 120.46s/it]

CPU times: total: 29min 9s
Wall time: 16min 31s


In [19]:
if 0 % 2:
    print(0)
else:
    print('ok')

ok


In [20]:
import pandas as pd

# Leggi il CSV
df = pd.read_csv(f'{fixed_path}CSNN/experiment/tuning_ball_logistic/results.csv')

# Visualizza tutto
print(df)


   seed negative_penalty    lr  dd_threshold  architecture  \
0     0           [3, 1]  95.0        0.0010             1   
1     0           [3, 1]  95.0        0.0001             1   
2     0           [3, 1]  99.9        0.0010             1   
3     0           [3, 1]  99.9        0.0001             1   
4     0           [6, 1]  95.0        0.0010             1   
5     0           [6, 1]  95.0        0.0001             1   
6     0           [6, 1]  99.9        0.0010             1   
7     0           [6, 1]  99.9        0.0001             1   

            param_hash  accuracy_train  f1_train  precision_train  \
0  8074551839613426537        0.666667       0.5              1.0   
1 -5509716523847018478        0.666667       0.5              1.0   
2 -1279874451465168874        0.833333       0.8              1.0   
3  8635098933196288869        0.833333       0.8              1.0   
4 -6786696229507795241        0.666667       0.5              1.0   
5  6251764899645875872     

In [21]:

# Visualizza le prime righe
print(df.head())



   seed negative_penalty    lr  dd_threshold  architecture  \
0     0           [3, 1]  95.0        0.0010             1   
1     0           [3, 1]  95.0        0.0001             1   
2     0           [3, 1]  99.9        0.0010             1   
3     0           [3, 1]  99.9        0.0001             1   
4     0           [6, 1]  95.0        0.0010             1   

            param_hash  accuracy_train  f1_train  precision_train  \
0  8074551839613426537        0.666667       0.5              1.0   
1 -5509716523847018478        0.666667       0.5              1.0   
2 -1279874451465168874        0.833333       0.8              1.0   
3  8635098933196288869        0.833333       0.8              1.0   
4 -6786696229507795241        0.666667       0.5              1.0   

   recall_train  auc_train  accuracy_valid  f1_valid  precision_valid  \
0      0.333333   0.777778        0.638889  0.434783             1.00   
1      0.333333   1.000000        0.722222  0.615385             1

In [22]:
# Visualizza statistiche
print(df.describe())



       seed        lr  dd_threshold  architecture    param_hash  \
count   8.0   8.00000      8.000000           8.0  8.000000e+00   
mean    0.0  97.45000      0.000550           1.0  1.905352e+17   
std     0.0   2.61916      0.000481           0.0  6.761291e+18   
min     0.0  95.00000      0.000100           1.0 -7.961376e+18   
25%     0.0  95.00000      0.000100           1.0 -5.828961e+18   
50%     0.0  97.45000      0.000550           1.0 -5.896726e+17   
75%     0.0  99.90000      0.001000           1.0  6.707462e+18   
max     0.0  99.90000      0.001000           1.0  8.635099e+18   

       accuracy_train  f1_train  precision_train  recall_train  auc_train  \
count        8.000000  8.000000              8.0      8.000000   8.000000   
mean         0.750000  0.650000              1.0      0.500000   0.916667   
std          0.089087  0.160357              0.0      0.178174   0.129441   
min          0.666667  0.500000              1.0      0.333333   0.666667   
25%        

In [23]:
# Trova il modello migliore per AUC validation
best_model = df.loc[df['auc_valid'].idxmax()]
print("\nBest model:")
print(best_model)

# recostruct path for the best model
base_path = f'{fixed_path}CSNN/experiment/tuning_ball_logistic/models/'
model_seed = best_model['seed']
model_hash = best_model['param_hash']
best_model_path_recostructed = f'{base_path}/seed_{model_seed}/hash_{model_hash}/'
print(best_model_path_recostructed)



Best model:
seed                                  0
negative_penalty                 [3, 1]
lr                                 99.9
dd_threshold                     0.0001
architecture                          1
param_hash          8635098933196288869
accuracy_train                 0.833333
f1_train                            0.8
precision_train                     1.0
recall_train                   0.666667
auc_train                           1.0
accuracy_valid                 0.666667
f1_valid                            0.6
precision_valid                    0.75
recall_valid                        0.5
auc_valid                       0.87037
Name: 3, dtype: object
C:\Users\admin\0.Master_Thesis\CSNN/experiment/tuning_ball_logistic/models//seed_0/hash_8635098933196288869/


In [24]:
# Ordina per una metrica specifica
df_sorted = df.sort_values('auc_valid', ascending=False)
print("\nTop 5 models:")
print(df_sorted.head())


Top 5 models:
   seed negative_penalty    lr  dd_threshold  architecture  \
3     0           [3, 1]  99.9        0.0001             1   
7     0           [6, 1]  99.9        0.0001             1   
6     0           [6, 1]  99.9        0.0010             1   
1     0           [3, 1]  95.0        0.0001             1   
5     0           [6, 1]  95.0        0.0001             1   

            param_hash  accuracy_train  f1_train  precision_train  \
3  8635098933196288869        0.833333       0.8              1.0   
7   100529230252847219        0.833333       0.8              1.0   
6 -7961376316289067772        0.833333       0.8              1.0   
1 -5509716523847018478        0.666667       0.5              1.0   
5  6251764899645875872        0.666667       0.5              1.0   

   recall_train  auc_train  accuracy_valid  f1_valid  precision_valid  \
3      0.666667        1.0        0.666667  0.600000         0.750000   
7      0.666667        1.0        0.722222  0.68750

In [25]:

from utils import trainer, DensityDifference, dataset


decache_files = ['tuning_ball_logistic.yaml', 'timepoints_elaboration', 'results_elaboration', 'new_datasets_generation',
                'utils.trainer', 'utils.DensityDifference', 'utils.dataset']


# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)

from utils import trainer, DensityDifference, dataset

from timepoints_elaboration import donation_extraction, dataset_elaboration, donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from new_datasets_generation import splitting_and_dataset_elaboration, remove_labels


tuning_ball_logistic.yaml non trovato nella cache
timepoints_elaboration rimosso dalla cache
results_elaboration rimosso dalla cache
new_datasets_generation rimosso dalla cache
utils.trainer rimosso dalla cache
utils.DensityDifference rimosso dalla cache
utils.dataset rimosso dalla cache


In [26]:
#dataset_test = PointsetDataset(config['datasets'], train=False, fold=0, random_state=seed) #, **config['dataset_config'])
#print(os.path.exists(f'{path_unfixed}/data/B-ALL/test'))

sys.argv = ['test_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\test_ball_logistic.yaml']
with open(sys.argv[1], 'r') as f:
    test_config = load(f, Loader=FullLoader)
    print(test_config)
#test_path = f'{path_unfixed}/data/B-ALL/test'
print(test_config['datasets'])


#FIXARE TESTING
dataset_test = PointsetDataset(test_config['datasets'], train=False, fold=None, random_state=seed, test = True, **test_config['dataset_config'])

for data in dataset_test:
    print(data[0].shape)

{'experiment_name': 'test_ball_logistic', 'start_seed': 0, 'n_seeds': 1, 'datasets': ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/'], 'dataset_config': {'sample_size': 100000, 'train_proportions': 0}}
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/']
33
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
torch.Size([100000, 11])
tor

In [27]:
import hashlib
def test_load_evaluate(dataset_test, fn):
    #print(dataset.shape)
    x_test = dataset_test.X
    #print(x_test.shape)
    y_test = dataset_test.y
    
    x_test = x_test.to(device).float()
    y_test = y_test.to(device).float()

    model = torch.load(fn)
    model = model.to(device)
    model.eval()

    with torch.set_grad_enabled(False):
        preds_test = model(x_test)
        preds_test = preds_test.squeeze(1)

    return model.cpu(), (preds_test.cpu(), y_test.cpu())
    

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


model, (preds_test, y_test) = test_load_evaluate(dataset_test,  f'{best_model_path_recostructed}model.pt')
print(f'Test Predictions: {preds_test}')

final_pred_labels = []
for pred in preds_test:
    if pred > 0.5:
        final_pred_labels.append(1)
    else:
        final_pred_labels.append(0)

final_y_test = []
for lab in y_test:
    final_y_test.append(int(lab.item()))
print(f'Test Predicted Labels: {final_pred_labels}')

correct_counter = 0
for pred, true in zip(final_pred_labels, final_y_test):
    if pred == true:
        correct_counter += 1
print(f'Accuracy: {correct_counter / len(final_y_test)}')
print(f'True Labels: {final_y_test}')
print(f'True Labels Tensor: {y_test}')
#print(new_test_y)
        



Test Predictions: tensor([0.5057, 0.5006, 0.4989, 0.4991, 0.4994, 0.4989, 0.5024, 0.4990, 0.4990,
        0.4991, 0.4991, 0.4998, 0.4989, 0.4992, 0.4992, 0.4991, 0.4989, 0.4989,
        0.4991, 0.4990, 0.4989, 0.4994, 0.4990, 0.5018, 0.5006, 0.4993, 0.5007,
        0.4993, 0.4993, 0.4990, 0.4990, 0.4990, 0.4991])
Test Predicted Labels: [1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0]
Accuracy: 0.7272727272727273
True Labels: [1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0]
True Labels Tensor: tensor([1., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0.,
        0., 0., 0., 1., 1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0.])


In [28]:
test_indices = ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\9.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\18.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\5.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\6.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\27.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\3.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\12.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\13.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\20.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\17.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\19.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\26.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\25.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\21.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\11.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\1.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\2.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\14.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\22.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\8.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\7.pt']

valid_indices = ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\24.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\15.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\23.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\16.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\4.pt', 'C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn\\0.Unfixed_files/data/B-ALL/train\\10.pt']
#def splitting_debug()
for fn in test_indices:
        
        if fn in test_indices and fn in valid_indices:
            counter += 1
            print(counter)


In [29]:
s = r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\0.Unfixed_files/data/B-ALL/train\0_1.pt'
def retrieve_patient_index(fn):
    id = fn[::-1].index("\\")
    fn = fn[::-1][:id][::-1]
    _id = fn.index('_')
    patient_index = fn[:_id]
    return patient_index
